In [2]:
"""
================================================================================
OVERNIGHT BATCH (≈32 runs) — Trailing Stop-Loss Tuning
================================================================================
Assumption: 1 run ~ 20 minutes
Target: 10–12 hours overnight => ~30–36 runs

This script runs a curated batch of configs (not a huge grid) that usually move:
- TrailingSL_PctImproved
- TrailingSL_AvgPctImpact
- Trigger rate

Saved outputs:
1) overnight_results.csv      -> compact table (best for morning review)
2) BEST_* files (top K)       -> trade-level outputs only for top runs

================================================================================
"""

import os
import time
import numpy as np
import pandas as pd
from dataclasses import dataclass
from datetime import timedelta

# ============================================================
# PATHS
# ============================================================
TRADE_CSV_PATH = r"D:/work/Client/Maatra/Trade Level Data/Equities_Trade_data_14vol_2025.csv"
MAPPING_XLSX_PATH = r"D:/work/Client/Maatra/Trade Level Data/CMC_Finalto Inst.xlsx"


RS15_ROOT = r"D:\work\Trade Analysis\Polygon_15min_from_5min"
RS15_FILE_NAME = "rs_15min.parquet"  # <-- change if your file is named differently

STEP8_DIR = r"D:/work/Client/Maatra/Trade Level Data/TrailingSL_CAPPED_MKT1430_2100_SME_15T_STEP8_DIRSPEC"
BEST_STEP8_PATH = os.path.join(STEP8_DIR, "best_config_step8.csv")

STEP9_DIR = r"D:/work/Client/Maatra/Trade Level Data/TrailingSL_CAPPED_MKT1430_2100_SME_15T_STEP9_SENSITIVITY"
BEST_STEP9_PATH = os.path.join(STEP9_DIR, "best_config_step9.csv")

YEAR_FILTER = 2025

OUT_DIR = os.path.join(os.path.dirname(TRADE_CSV_PATH), f"TrailingSL_OVERNIGHT_BATCH_{YEAR_FILTER}")
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# BASE LOGIC
# ============================================================
ATR_PERIOD = 14
REG_WINDOW = 16
ATR_FALLBACK_MULT = 3.0

TRIGGER_ON_CLOSE = True          # confirmation works for close-based
INITIAL_MIN_STOP_PCT = 0.01      # you can also test 0.015/0.02 but you said 2% didn't help much
MIN_STOP_PCT = 0.0010

# ============================================================
# RUN CONTROL
# ============================================================
MAX_HOURS = 12.0
SAVE_FULL_OUTPUT_FOR_BEST_K = 5

# Rank by what you care about most:
# 1) TrailingSL_PctImproved => pushes the number you want
# 2) TrailingSL_AvgPctImpact => ensures improvement is meaningful
RANK_BY_PRIMARY = "TrailingSL_PctImproved"
RANK_BY_SECONDARY = "TrailingSL_AvgPctImpact"

# ============================================================
# 32 curated configs (≈10–12 hours if 1 run ~20 min)
# Each config tests one strong lever at a time:
#  - grace (avoid early noise)
#  - trailing delay (don’t tighten too soon)
#  - throttle updates (avoid over-tightening)
#  - confirm bars (avoid whipsaws)
#  - accel cap (avoid aggressive tightening spikes)
#  - loosen shorts slightly
# ============================================================
RUNS = [
    # --- Baseline-ish ---
    dict(G=0,  TS=0,  TU=1, C=1, AC=1.0, SL=None, SS=None, name="BASELINE"),
    dict(G=0,  TS=8,  TU=2, C=1, AC=1.0, SL=None, SS=None, name="MILD_DELAY"),

    # --- Grace period sweep ---
    dict(G=2,  TS=0,  TU=1, C=1, AC=1.0, SL=None, SS=None, name="GRACE_30M"),
    dict(G=4,  TS=0,  TU=1, C=1, AC=1.0, SL=None, SS=None, name="GRACE_60M"),
    dict(G=6,  TS=0,  TU=1, C=1, AC=1.0, SL=None, SS=None, name="GRACE_90M"),

    # --- Trailing start delay sweep ---
    dict(G=0,  TS=8,  TU=1, C=1, AC=1.0, SL=None, SS=None, name="TRAIL_START_2H"),
    dict(G=0,  TS=12, TU=1, C=1, AC=1.0, SL=None, SS=None, name="TRAIL_START_3H"),
    dict(G=0,  TS=16, TU=1, C=1, AC=1.0, SL=None, SS=None, name="TRAIL_START_4H"),

    # --- Update throttle sweep ---
    dict(G=0,  TS=0,  TU=2, C=1, AC=1.0, SL=None, SS=None, name="UPDATE_30M"),
    dict(G=0,  TS=0,  TU=4, C=1, AC=1.0, SL=None, SS=None, name="UPDATE_60M"),
    dict(G=0,  TS=0,  TU=6, C=1, AC=1.0, SL=None, SS=None, name="UPDATE_90M"),

    # --- Confirmation sweep (close-based) ---
    dict(G=0,  TS=0,  TU=1, C=2, AC=1.0, SL=None, SS=None, name="CONFIRM_2"),
    dict(G=0,  TS=0,  TU=1, C=3, AC=1.0, SL=None, SS=None, name="CONFIRM_3"),

    # --- Accel cap sweep ---
    dict(G=0,  TS=0,  TU=1, C=1, AC=0.25, SL=None, SS=None, name="ACCELCAP_0.25"),
    dict(G=0,  TS=0,  TU=1, C=1, AC=0.50, SL=None, SS=None, name="ACCELCAP_0.50"),

    # --- Combined “likely winners” (grace + delay + throttle + confirm + accel cap) ---
    dict(G=4,  TS=12, TU=4, C=2, AC=0.50, SL=None, SS=None, name="COMBO_A"),
    dict(G=4,  TS=16, TU=4, C=2, AC=0.50, SL=None, SS=None, name="COMBO_B"),
    dict(G=6,  TS=12, TU=6, C=2, AC=0.50, SL=None, SS=None, name="COMBO_C"),
    dict(G=6,  TS=16, TU=6, C=2, AC=0.25, SL=None, SS=None, name="COMBO_D"),

    # --- Shorts loosened (often improves trailing behavior on shorts) ---
    dict(G=4,  TS=12, TU=4, C=2, AC=0.50, SL=None, SS=0.85, name="COMBO_A_SHORTS0.85"),
    dict(G=4,  TS=12, TU=4, C=2, AC=0.50, SL=None, SS=0.80, name="COMBO_A_SHORTS0.80"),
    dict(G=6,  TS=16, TU=6, C=2, AC=0.25, SL=None, SS=0.80, name="COMBO_D_SHORTS0.80"),

    # --- Add a couple “aggressive” variants just for contrast ---
    dict(G=0,  TS=0,  TU=1, C=1, AC=2.0, SL=None, SS=None, name="AGGR_ACCEL_2.0"),
    dict(G=0,  TS=0,  TU=1, C=1, AC=1.0, SL=1.1, SS=None, name="SENS_LONG_1.1"),
    dict(G=0,  TS=0,  TU=1, C=1, AC=1.0, SL=0.9, SS=None, name="SENS_LONG_0.9"),

    # --- Extra combos to reach ~32 runs ---
    dict(G=2,  TS=12, TU=4, C=2, AC=0.50, SL=None, SS=None, name="COMBO_E"),
    dict(G=2,  TS=16, TU=4, C=2, AC=0.50, SL=None, SS=None, name="COMBO_F"),
    dict(G=4,  TS=8,  TU=4, C=2, AC=0.50, SL=None, SS=None, name="COMBO_G"),
    dict(G=4,  TS=12, TU=2, C=2, AC=0.50, SL=None, SS=None, name="COMBO_H"),
    dict(G=6,  TS=12, TU=4, C=3, AC=0.50, SL=None, SS=None, name="COMBO_I"),
    dict(G=6,  TS=16, TU=4, C=3, AC=0.25, SL=None, SS=None, name="COMBO_J"),
    dict(G=4,  TS=16, TU=2, C=2, AC=0.25, SL=None, SS=None, name="COMBO_K"),
    dict(G=2,  TS=8,  TU=6, C=2, AC=0.50, SL=None, SS=None, name="COMBO_L"),
]

# ============================================================
# Utility
# ============================================================
def read_trade_csv(path: str) -> pd.DataFrame:
    for enc in ["utf-8", "cp1252", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, encoding="utf-8", encoding_errors="ignore", low_memory=False)

def normalize_ticker(x: str) -> str:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    x = str(x).strip().upper()
    return "".join([ch for ch in x if ch.isalnum()])

def trade_base_from_currency(currency: str) -> str:
    if currency is None or (isinstance(currency, float) and np.isnan(currency)):
        return ""
    s = str(currency).strip().upper()
    base = s.split("/")[0].strip() if "/" in s else s
    return normalize_ticker(base)

def normalize_direction(x) -> str:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "UNKNOWN"
    s = str(x).strip().upper()
    if s in ["LONG","BUY","B","1","1.0","+1","+1.0"]:
        return "LONG"
    if s in ["SHORT","SELL","S","-1","-1.0"]:
        return "SHORT"
    try:
        v = float(s)
        if v > 0: return "LONG"
        if v < 0: return "SHORT"
    except Exception:
        pass
    return "UNKNOWN"

def tz_naive_utc(s: pd.Series) -> pd.Series:
    if not pd.api.types.is_datetime64_any_dtype(s):
        s = pd.to_datetime(s, errors="coerce")
    try:
        tz = getattr(s.dt, "tz", None)
    except Exception:
        tz = None
    if tz is not None:
        s = s.dt.tz_convert("UTC").dt.tz_localize(None)
    return s

def load_trade_to_rs15_mapping(mapping_xlsx_path: str) -> dict:
    mp = pd.read_excel(mapping_xlsx_path)
    mp.columns = [c.strip() for c in mp.columns]
    if "org_symbol" not in mp.columns or "Symbol" not in mp.columns:
        raise ValueError("Mapping file must contain columns: org_symbol, Symbol")
    mp["rs15_ticker"] = mp["org_symbol"].apply(normalize_ticker)
    mp["trade_base"] = mp["Symbol"].astype(str).str.upper().str.strip().str.split("/").str[0].apply(normalize_ticker)
    mp = mp[(mp["trade_base"]!="") & (mp["rs15_ticker"]!="")]
    mp = mp.drop_duplicates(subset=["trade_base"], keep="first")
    return dict(zip(mp["trade_base"], mp["rs15_ticker"]))

TRADE_TO_RS15 = load_trade_to_rs15_mapping(MAPPING_XLSX_PATH)

def map_trade_base_to_rs15(trade_base: str) -> str:
    t = normalize_ticker(trade_base)
    return TRADE_TO_RS15.get(t, t)

def load_rs15_tickers(rs15_root: str) -> set:
    out=set()
    for name in os.listdir(rs15_root):
        if name.lower().startswith("symbol="):
            out.add(normalize_ticker(name.split("=",1)[1]))
    return out

RS15_TICKERS = load_rs15_tickers(RS15_ROOT)

def compute_atr(df: pd.DataFrame, period: int) -> pd.Series:
    high = df["high"]
    low = df["low"]
    close = df["close"]
    tr = pd.concat([high-low,(high-close.shift(1)).abs(),(low-close.shift(1)).abs()], axis=1).max(axis=1)
    return tr.rolling(period, min_periods=period).mean()

_RS_CACHE = {}
def prepare_rs_for_ticker(ticker: str):
    t = normalize_ticker(ticker)
    if t in _RS_CACHE:
        return _RS_CACHE[t]
    fp = os.path.join(RS15_ROOT, f"symbol={t}", RS15_FILE_NAME)
    if not os.path.exists(fp):
        _RS_CACHE[t] = None
        return None
    rs = pd.read_parquet(fp)
    if rs is None or rs.empty:
        _RS_CACHE[t] = None
        return None
    rs.columns = [c.strip() for c in rs.columns]
    need = ["_dt","open","high","low","close"]
    if not all(c in rs.columns for c in need):
        _RS_CACHE[t] = None
        return None
    rs["_dt"] = tz_naive_utc(rs["_dt"])
    rs = rs.dropna(subset=["_dt"]).sort_values("_dt").reset_index(drop=True)
    for c in ["open","high","low","close"]:
        rs[c] = pd.to_numeric(rs[c], errors="coerce")
    rs = rs.dropna(subset=["open","high","low","close"])
    if rs.empty:
        _RS_CACHE[t] = None
        return None
    rs["ATR"] = compute_atr(rs, ATR_PERIOD)
    _RS_CACHE[t] = rs
    return rs

def load_best_step8(path: str) -> dict:
    b8 = pd.read_csv(path).iloc[0].to_dict()
    return {k: float(b8[k]) for k in ["ATAN_LONG","ATAN_SHORT","MAX_STOP_LONG","MAX_STOP_SHORT"]}

def load_best_step9(path: str) -> dict:
    b9 = pd.read_csv(path).iloc[0].to_dict()
    return {"SENS_LONG": float(b9.get("SENS_LONG",1.0)), "SENS_SHORT": float(b9.get("SENS_SHORT",1.0))}

BASE_CFG = {**load_best_step8(BEST_STEP8_PATH), **load_best_step9(BEST_STEP9_PATH)}

def build_trade_summary(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.copy()
    df.columns = [c.strip() for c in df.columns]
    req = ["TradeID","Currency","Direction","date","price","Opening Date","Closing Date"]
    miss = [c for c in req if c not in df.columns]
    if miss: raise ValueError(f"Missing columns: {miss}")

    df["date"] = tz_naive_utc(pd.to_datetime(df["date"], errors="coerce"))
    df["Opening Date"] = tz_naive_utc(pd.to_datetime(df["Opening Date"], errors="coerce"))
    df["Closing Date"] = tz_naive_utc(pd.to_datetime(df["Closing Date"], errors="coerce"))
    df["price"] = pd.to_numeric(df["price"], errors="coerce")

    df = df[df["date"].dt.year == YEAR_FILTER].copy()
    df["TradeBase"] = df["Currency"].apply(trade_base_from_currency)
    df["TickerNorm"] = df["TradeBase"].apply(map_trade_base_to_rs15)
    df["DirectionNorm"] = df["Direction"].apply(normalize_direction)

    rows=[]
    for tid, g in df.groupby("TradeID", dropna=True):
        g = g.sort_values("date")
        entry_date = g["Opening Date"].dropna().iloc[0] if g["Opening Date"].notna().any() else pd.NaT
        exit_date  = g["Closing Date"].dropna().iloc[0] if g["Closing Date"].notna().any() else pd.NaT
        direction  = g["DirectionNorm"].dropna().iloc[0] if g["DirectionNorm"].notna().any() else "UNKNOWN"
        ticker     = g["TickerNorm"].dropna().iloc[0] if g["TickerNorm"].notna().any() else ""

        entry_px=np.nan
        if pd.notna(entry_date):
            r = g[g["date"].dt.normalize()==entry_date.normalize()]
            if r["price"].notna().any(): entry_px=float(r["price"].dropna().iloc[-1])

        exit_px=np.nan
        if pd.notna(exit_date):
            r = g[g["date"].dt.normalize()==exit_date.normalize()]
            if r["price"].notna().any(): exit_px=float(r["price"].dropna().iloc[-1])

        valid_window = pd.notna(entry_date) and pd.notna(exit_date) and (exit_date.normalize()>entry_date.normalize())
        core_ok = (ticker!="" and ticker in RS15_TICKERS and direction!="UNKNOWN" and
                   pd.notna(entry_date) and pd.notna(exit_date) and pd.notna(entry_px) and pd.notna(exit_px))

        rows.append({
            "TradeID":tid,"TickerNorm":ticker,"Direction":direction,
            "EntryDate":entry_date,"ExitDate_Original":exit_date,
            "EntryPrice":entry_px,"ExitPrice_Original":exit_px,
            "HasAllCoreFields":bool(core_ok),"HasValidTrailingWindow":bool(valid_window)
        })
    return pd.DataFrame(rows)

@dataclass
class TrailResult:
    exit_time: pd.Timestamp
    exit_price: float
    exit_reason: str

def adaptive_stop_distance_pct(bars, atan_thr, min_stop, max_stop, sens, accel_cap):
    if len(bars) < REG_WINDOW or bars["ATR"].dropna().empty:
        return np.nan
    recent = bars["close"].iloc[-REG_WINDOW:].astype(float)
    if len(recent) < REG_WINDOW:
        return np.nan
    x = np.arange(len(recent), dtype=float)
    a, b, _ = np.polyfit(x, recent.values, 2)
    slope_norm = (2*a*x[-1]+b)/recent.iloc[-1]
    accel = 2*a
    atan_slope = float(np.arctan(slope_norm))

    atr = bars["ATR"].iloc[-1]
    px = bars["close"].iloc[-1]
    atr_pct = float(atr/px) if (pd.notna(atr) and atr>0 and px>0) else np.nan
    if pd.isna(atr_pct):
        return np.nan

    if abs(atan_slope) > atan_thr:
        accel_factor = 1.0 + min(abs(accel), float(accel_cap))
        stop_pct = abs(slope_norm) * sens * accel_factor
        return float(np.clip(stop_pct, min_stop, max_stop))

    stop_pct = atr_pct * ATR_FALLBACK_MULT
    return float(np.clip(stop_pct, min_stop, max_stop))

def stop_hit_confirmed(sub, is_long, stop, confirm_bars):
    k = int(max(1, confirm_bars))
    if len(sub) < k: return False
    closes = sub["close"].iloc[-k:].astype(float)
    return bool((closes <= stop).all()) if is_long else bool((closes >= stop).all())

def price_improvement(direction, new_exit, old_exit):
    return (new_exit-old_exit) if normalize_direction(direction)=="LONG" else (old_exit-new_exit)

def simulate_trade(tr, rs, cfg, G, TS, TU, C, AC):
    entry_date = pd.to_datetime(tr["EntryDate"])
    exit_date  = pd.to_datetime(tr["ExitDate_Original"])
    entry_px   = float(tr["EntryPrice"])
    orig_exit  = float(tr["ExitPrice_Original"])

    direction = normalize_direction(tr["Direction"])
    is_long = (direction=="LONG")

    # Initial SL (keep as 1% floor)
    atan_thr = cfg["ATAN_LONG"] if is_long else cfg["ATAN_SHORT"]
    max_stop = cfg["MAX_STOP_LONG"] if is_long else cfg["MAX_STOP_SHORT"]
    sens     = cfg["SENS_LONG"] if is_long else cfg["SENS_SHORT"]

    init_hist = rs[rs["_dt"] <= (entry_date.normalize() + pd.Timedelta(hours=23,minutes=59,seconds=59))]
    init_stop_pct = adaptive_stop_distance_pct(init_hist, atan_thr, INITIAL_MIN_STOP_PCT, max_stop, sens, AC)
    if pd.isna(init_stop_pct):
        init_stop_pct = INITIAL_MIN_STOP_PCT

    stop = entry_px*(1.0-init_stop_pct) if is_long else entry_px*(1.0+init_stop_pct)
    best_close = entry_px
    stop_source = "Initial"

    w = rs[(rs["_dt"] >= (entry_date.normalize()+timedelta(days=1))) & (rs["_dt"] < (exit_date+timedelta(days=1)))].copy()
    if w.empty:
        return TrailResult(exit_date, orig_exit, "NoRSDataInWindow")

    for i in range(len(w)):
        sub = w.iloc[:i+1]
        close = float(w.iloc[i]["close"])
        dt = w.iloc[i]["_dt"]

        best_close = max(best_close, close) if is_long else min(best_close, close)

        if i < int(G):
            continue

        # Stop check (confirmed)
        if TRIGGER_ON_CLOSE:
            if stop_hit_confirmed(sub, is_long, stop, int(C)):
                return TrailResult(dt, float(stop), "InitialSL" if stop_source=="Initial" else "TrailingSL")

        # Trailing updates
        if i < int(TS):
            continue
        every = int(max(1, TU))
        if every > 1 and (i % every) != 0:
            continue

        stop_pct = adaptive_stop_distance_pct(sub, atan_thr, MIN_STOP_PCT, max_stop, sens, AC)
        if pd.isna(stop_pct):
            continue

        if is_long:
            stop = max(stop, best_close*(1.0-stop_pct))
        else:
            stop = min(stop, best_close*(1.0+stop_pct))
        stop_source = "Trailing"

    return TrailResult(exit_date, orig_exit, "OriginalExit")

def metrics(out_df):
    df = out_df.copy()
    df["Triggered"] = df["ExitReason"].isin(["TrailingSL","InitialSL"])
    df["Improved"] = df["PriceImprovement"] > 0
    tsl = df[df["ExitReason"]=="TrailingSL"]
    trig = df[df["Triggered"]]
    return {
        "Trades": len(df),
        "SLTriggered": int(df["Triggered"].sum()),
        "TrailingSL_Trades": int((df["ExitReason"]=="TrailingSL").sum()),
        "TrailingSL_PctImproved": float(tsl["Improved"].mean()) if len(tsl) else np.nan,
        "TrailingSL_AvgPctImpact": float(tsl["PctImprovement_vs_OriginalExit"].mean()) if len(tsl) else np.nan,
        "Triggered_PctImproved": float(trig["Improved"].mean()) if len(trig) else np.nan,
        "Triggered_AvgPctImpact": float(trig["PctImprovement_vs_OriginalExit"].mean()) if len(trig) else np.nan,
    }

def main():
    t0 = time.time()

    print("Loading trades once...")
    raw = read_trade_csv(TRADE_CSV_PATH)
    trades = build_trade_summary(raw)
    eligible = trades[(trades["HasAllCoreFields"]) & (trades["HasValidTrailingWindow"])].copy()
    eligible = eligible.sort_values(["TickerNorm","EntryDate","TradeID"]).reset_index(drop=True)

    print("Eligible trades:", len(eligible), "| unique tickers:", eligible["TickerNorm"].nunique())

    # preload rs15
    tickers = sorted(eligible["TickerNorm"].unique())
    print("Preloading rs15...")
    for i, t in enumerate(tickers, 1):
        prepare_rs_for_ticker(t)
        if i % 100 == 0:
            print(f"  loaded {i}/{len(tickers)}")

    results=[]
    best=[]

    for idx, r in enumerate(RUNS, 1):
        elapsed = (time.time()-t0)/3600.0
        if elapsed >= MAX_HOURS:
            print(f"Stopping due to MAX_HOURS={MAX_HOURS}")
            break

        cfg = dict(BASE_CFG)
        if r["SL"] is not None: cfg["SENS_LONG"] = float(r["SL"])
        if r["SS"] is not None: cfg["SENS_SHORT"] = float(r["SS"])

        out=[]
        for _, tr in eligible.iterrows():
            rs = prepare_rs_for_ticker(tr["TickerNorm"])
            if rs is None or rs.empty:
                continue
            res = simulate_trade(tr, rs, cfg, r["G"], r["TS"], r["TU"], r["C"], r["AC"])
            old_exit = float(tr["ExitPrice_Original"])
            new_exit = float(res.exit_price)
            impr = price_improvement(tr["Direction"], new_exit, old_exit)
            out.append({
                "TradeID": tr["TradeID"],
                "ExitReason": res.exit_reason,
                "PriceImprovement": impr,
                "PctImprovement_vs_OriginalExit": (impr/old_exit) if old_exit else np.nan,
            })

        out_df = pd.DataFrame(out)
        m = metrics(out_df)

        row = {
            "RunIdx": idx,
            "Name": r["name"],
            "G": r["G"], "TS": r["TS"], "TU": r["TU"], "C": r["C"], "AC": r["AC"],
            "SENS_LONG": cfg["SENS_LONG"], "SENS_SHORT": cfg["SENS_SHORT"],
            "ElapsedHours": round((time.time()-t0)/3600.0, 4),
            **m,
        }
        results.append(row)

        # ranking
        score = row.get(RANK_BY_PRIMARY, np.nan)
        tie   = row.get(RANK_BY_SECONDARY, np.nan)
        if pd.notna(score):
            best.append((float(score), float(tie) if pd.notna(tie) else -1e9, row, out_df))
            best.sort(key=lambda x: (x[0], x[1]), reverse=True)
            best = best[:SAVE_FULL_OUTPUT_FOR_BEST_K]

        print(f"[{idx}/{len(RUNS)}] {r['name']} | {RANK_BY_PRIMARY}={row.get(RANK_BY_PRIMARY)} | TrailingSL_Trades={row['TrailingSL_Trades']}")

    res_df = pd.DataFrame(results)
    res_fp = os.path.join(OUT_DIR, "overnight_results.csv")
    res_df.to_csv(res_fp, index=False)

    # save best full outputs
    for k, (score, tie, row, out_df) in enumerate(best, 1):
        fp = os.path.join(OUT_DIR, f"BEST_{k}_{RANK_BY_PRIMARY}_{score:.4f}_{row['Name']}.csv")
        out_df.to_csv(fp, index=False)

    print("\n✅ Saved:", res_fp)
    if best:
        print("Top runs:")
        for k, (score, tie, row, _) in enumerate(best, 1):
            print(f"  {k}. {row['Name']} | {RANK_BY_PRIMARY}={score:.4f} | {RANK_BY_SECONDARY}={tie:.4f}")

if __name__ == "__main__":
    main()

Loading trades once...
Eligible trades: 135785 | unique tickers: 811
Preloading rs15...
  loaded 100/811
  loaded 200/811
  loaded 300/811
  loaded 400/811
  loaded 500/811
  loaded 600/811
  loaded 700/811
  loaded 800/811
[1/33] BASELINE | TrailingSL_PctImproved=0.5446158409967369 | TrailingSL_Trades=67420
[2/33] MILD_DELAY | TrailingSL_PctImproved=0.5496969603503847 | TrailingSL_Trades=64843
[3/33] GRACE_30M | TrailingSL_PctImproved=0.5438077686824098 | TrailingSL_Trades=69999
[4/33] GRACE_60M | TrailingSL_PctImproved=0.541725719049604 | TrailingSL_Trades=71970
[5/33] GRACE_90M | TrailingSL_PctImproved=0.5401508279288916 | TrailingSL_Trades=73859
[6/33] TRAIL_START_2H | TrailingSL_PctImproved=0.5446158409967369 | TrailingSL_Trades=67420
[7/33] TRAIL_START_3H | TrailingSL_PctImproved=0.5446158409967369 | TrailingSL_Trades=67420
[8/33] TRAIL_START_4H | TrailingSL_PctImproved=0.5502651129578706 | TrailingSL_Trades=65821
[9/33] UPDATE_30M | TrailingSL_PctImproved=0.5496969603503847 | Tr

In [4]:
"""
================================================================================
FINAL TRADE-LEVEL OUTPUT (Best Config = COMBO_J) — Portfolio-Ready Export
================================================================================
Goal
----
Generate a FINAL trade-level file for YEAR_FILTER using your best trailing SL setup
(COMBO_J), so you can build a portfolio and evaluate portfolio-level impact.

What this script does
---------------------
1) Loads trade CSV (your schema) and filters to YEAR_FILTER
2) Maps each trade's base symbol to RS15 ticker using Instrument Mapping.xlsx
3) Loads 15-min resampled parquet per ticker from:
      RS15_ROOT\symbol=XXXX\rs_15min.parquet
   (cached for speed) and computes ATR14
4) Runs trailing SL simulation per trade using:
      COMBO_J: G=6, TS=16, TU=4, C=3, AC=0.25, SENS_LONG=0.9, SENS_SHORT=0.9
   + Step8 base params for atan/max_stop (best_config_step8.csv)
5) Saves portfolio-ready outputs:
   - final_trades_{YEAR}_COMBO_J.csv (trade-level)
   - final_summary_{YEAR}_COMBO_J.csv
   - final_reason_counts_{YEAR}_COMBO_J.csv

Key Output Columns (trade-level)
--------------------------------
- EntryDate, EntryDate_Norm, EntryPrice
- ExitDate_Original, ExitDate_Original_Norm, ExitPrice_Original
- ExitTime_Trailing, ExitDate_Trailing_Norm, ExitPrice_Trailing, ExitReason
- ExitSameDayFlag, ExitDateChangedFlag, ExitTimeChangedFlag
- HoldingDays_Original, HoldingDays_Trailing
- ImprovedFlag, PriceImprovement, PctImprovement_vs_OriginalExit
- InitialStopPct, InitialStopPrice
- Params used (G,TS,TU,C,AC,SENS_LONG,SENS_SHORT)

Notes
-----
- RS15 file name confirmed by you: "rs_15min.parquet"
- Datetimes are made tz-naive (UTC) to avoid tz-aware/naive comparison errors.

================================================================================
"""

import os
import numpy as np
import pandas as pd
from dataclasses import dataclass
from datetime import timedelta

# ============================================================
# CONFIG — PATHS
# ============================================================
TRADE_CSV_PATH = r"D:/work/Client/Maatra/Trade Level Data/Equities_Trade_data_14vol_2025.csv"
MAPPING_XLSX_PATH = r"D:/work/Client/Maatra/Trade Level Data/CMC_Finalto Inst.xlsx"

RS15_ROOT = r"D:\work\Trade Analysis\Polygon_15min_from_5min"
RS15_FILE_NAME = "rs_15min.parquet"   # ✅ confirmed

# Step8 base params (direction-specific)
STEP8_DIR = r"D:/work/Client/Maatra/Trade Level Data/TrailingSL_CAPPED_MKT1430_2100_SME_15T_STEP8_DIRSPEC"
BEST_STEP8_PATH = os.path.join(STEP8_DIR, "best_config_step8.csv")

YEAR_FILTER = 2025

OUT_DIR = os.path.join(os.path.dirname(TRADE_CSV_PATH), f"TrailingSL_FINAL_{YEAR_FILTER}_COMBO_J")
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# BEST CONFIG (COMBO_J)
# ============================================================
BEST = {
    "GRACE_BARS": 6,                 # G (6*15=90 mins)
    "TRAILING_START_BARS": 16,       # TS (16*15=240 mins)
    "TRAILING_UPDATE_EVERY_BARS": 4, # TU (4*15=60 mins)
    "STOP_CONFIRM_BARS": 3,          # C
    "ACCEL_CAP": 0.25,               # AC
    "SENS_LONG": 0.9,
    "SENS_SHORT": 0.9,
}

# ============================================================
# Base logic
# ============================================================
ATR_PERIOD = 14
REG_WINDOW = 16
ATR_FALLBACK_MULT = 3.0

TRIGGER_ON_CLOSE = True  # close-based stop confirmation

INITIAL_MIN_STOP_PCT = 0.01  # 1%
MIN_STOP_PCT = 0.0010        # trailing min floor

PRINT_EVERY = 2000

# ============================================================
# Robust CSV reader
# ============================================================
def read_trade_csv(path: str) -> pd.DataFrame:
    for enc in ["utf-8", "cp1252", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, encoding="utf-8", encoding_errors="ignore", low_memory=False)

# ============================================================
# Helpers
# ============================================================
def normalize_ticker(x: str) -> str:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    x = str(x).strip().upper()
    return "".join([ch for ch in x if ch.isalnum()])

def trade_base_from_currency(currency: str) -> str:
    if currency is None or (isinstance(currency, float) and np.isnan(currency)):
        return ""
    s = str(currency).strip().upper()
    base = s.split("/")[0].strip() if "/" in s else s
    return normalize_ticker(base)

def normalize_direction(x) -> str:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "UNKNOWN"
    s = str(x).strip().upper()
    if s in ["LONG","BUY","B","1","1.0","+1","+1.0"]:
        return "LONG"
    if s in ["SHORT","SELL","S","-1","-1.0"]:
        return "SHORT"
    try:
        v = float(s)
        if v > 0: return "LONG"
        if v < 0: return "SHORT"
    except Exception:
        pass
    return "UNKNOWN"

def tz_naive_utc(s: pd.Series) -> pd.Series:
    """Convert tz-aware datetime series to UTC tz-naive; leave tz-naive as-is."""
    if not pd.api.types.is_datetime64_any_dtype(s):
        s = pd.to_datetime(s, errors="coerce")
    try:
        tz = getattr(s.dt, "tz", None)
    except Exception:
        tz = None
    if tz is not None:
        s = s.dt.tz_convert("UTC").dt.tz_localize(None)
    return s

# ============================================================
# Mapping: TradeBase -> RS15 ticker
# ============================================================
def load_trade_to_rs15_mapping(mapping_xlsx_path: str) -> dict:
    mp = pd.read_excel(mapping_xlsx_path)
    mp.columns = [c.strip() for c in mp.columns]
    if "org_symbol" not in mp.columns or "Symbol" not in mp.columns:
        raise ValueError("Mapping file must contain columns: org_symbol, Symbol")
    mp["rs15_ticker"] = mp["org_symbol"].apply(normalize_ticker)
    mp["trade_base"] = (
        mp["Symbol"].astype(str).str.upper().str.strip().str.split("/").str[0].apply(normalize_ticker)
    )
    mp = mp[(mp["trade_base"] != "") & (mp["rs15_ticker"] != "")]
    mp = mp.drop_duplicates(subset=["trade_base"], keep="first")
    return dict(zip(mp["trade_base"], mp["rs15_ticker"]))

TRADE_TO_RS15 = load_trade_to_rs15_mapping(MAPPING_XLSX_PATH)

def map_trade_base_to_rs15(trade_base: str) -> str:
    t = normalize_ticker(trade_base)
    return TRADE_TO_RS15.get(t, t)

def load_rs15_tickers(rs15_root: str) -> set:
    out = set()
    if not os.path.exists(rs15_root):
        return out
    for name in os.listdir(rs15_root):
        if name.lower().startswith("symbol="):
            out.add(normalize_ticker(name.split("=", 1)[1]))
    return out

RS15_TICKERS = load_rs15_tickers(RS15_ROOT)

# ============================================================
# RS15 load + ATR (cached)
# ============================================================
def compute_atr(df: pd.DataFrame, period: int) -> pd.Series:
    high = df["high"]
    low = df["low"]
    close = df["close"]
    tr = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low - close.shift(1)).abs()
    ], axis=1).max(axis=1)
    return tr.rolling(period, min_periods=period).mean()

_RS_CACHE = {}

def prepare_rs_for_ticker(ticker: str):
    t = normalize_ticker(ticker)
    if t in _RS_CACHE:
        return _RS_CACHE[t]

    fp = os.path.join(RS15_ROOT, f"symbol={t}", RS15_FILE_NAME)
    if not os.path.exists(fp):
        _RS_CACHE[t] = None
        return None

    rs = pd.read_parquet(fp)
    if rs is None or rs.empty:
        _RS_CACHE[t] = None
        return None

    rs.columns = [c.strip() for c in rs.columns]
    need = ["_dt", "open", "high", "low", "close"]
    if not all(c in rs.columns for c in need):
        _RS_CACHE[t] = None
        return None

    rs["_dt"] = tz_naive_utc(rs["_dt"])
    rs = rs.dropna(subset=["_dt"]).sort_values("_dt").reset_index(drop=True)

    for c in ["open", "high", "low", "close"]:
        rs[c] = pd.to_numeric(rs[c], errors="coerce")
    rs = rs.dropna(subset=["open", "high", "low", "close"])

    if rs.empty:
        _RS_CACHE[t] = None
        return None

    rs["ATR"] = compute_atr(rs, ATR_PERIOD)

    _RS_CACHE[t] = rs
    return rs

# ============================================================
# Trade extraction (your CSV schema)
# ============================================================
def build_trade_summary(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.copy()
    df.columns = [c.strip() for c in df.columns]

    required = ["TradeID", "Currency", "Direction", "date", "price", "Opening Date", "Closing Date"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Trade file missing required columns: {missing}")

    df["date"] = tz_naive_utc(pd.to_datetime(df["date"], errors="coerce"))
    df["Opening Date"] = tz_naive_utc(pd.to_datetime(df["Opening Date"], errors="coerce"))
    df["Closing Date"] = tz_naive_utc(pd.to_datetime(df["Closing Date"], errors="coerce"))
    df["price"] = pd.to_numeric(df["price"], errors="coerce")

    df = df[df["date"].dt.year == YEAR_FILTER].copy()

    df["TradeBase"] = df["Currency"].apply(trade_base_from_currency)
    df["TickerNorm"] = df["TradeBase"].apply(map_trade_base_to_rs15)
    df["DirectionNorm"] = df["Direction"].apply(normalize_direction)

    rows = []
    for tid, g in df.groupby("TradeID", dropna=True):
        g = g.sort_values("date")

        entry_date = g["Opening Date"].dropna().iloc[0] if g["Opening Date"].notna().any() else pd.NaT
        exit_date  = g["Closing Date"].dropna().iloc[0] if g["Closing Date"].notna().any() else pd.NaT

        direction = g["DirectionNorm"].dropna().iloc[0] if g["DirectionNorm"].notna().any() else "UNKNOWN"
        ticker = g["TickerNorm"].dropna().iloc[0] if g["TickerNorm"].notna().any() else ""

        entry_px = np.nan
        if pd.notna(entry_date):
            day_rows = g[g["date"].dt.normalize() == entry_date.normalize()]
            if day_rows["price"].notna().any():
                entry_px = float(day_rows["price"].dropna().iloc[-1])

        exit_px = np.nan
        if pd.notna(exit_date):
            day_rows = g[g["date"].dt.normalize() == exit_date.normalize()]
            if day_rows["price"].notna().any():
                exit_px = float(day_rows["price"].dropna().iloc[-1])

        valid_window = pd.notna(entry_date) and pd.notna(exit_date) and (exit_date.normalize() > entry_date.normalize())
        core_ok = (
            ticker != "" and ticker in RS15_TICKERS and
            direction != "UNKNOWN" and
            pd.notna(entry_date) and pd.notna(exit_date) and
            pd.notna(entry_px) and pd.notna(exit_px)
        )

        rows.append({
            "TradeID": tid,
            "Currency": g["Currency"].dropna().iloc[0] if g["Currency"].notna().any() else "",
            "TradeBase": trade_base_from_currency(g["Currency"].dropna().iloc[0]) if g["Currency"].notna().any() else "",
            "TickerNorm": ticker,
            "Direction": direction,
            "EntryDate": entry_date,
            "ExitDate_Original": exit_date,
            "EntryPrice": entry_px,
            "ExitPrice_Original": exit_px,
            "HasAllCoreFields": bool(core_ok),
            "HasValidTrailingWindow": bool(valid_window),
        })
    return pd.DataFrame(rows)

# ============================================================
# Simulation
# ============================================================
@dataclass
class TrailResult:
    exit_time: pd.Timestamp
    exit_price: float
    exit_reason: str
    init_stop_pct: float
    init_stop_price: float

def adaptive_stop_distance_pct(bars: pd.DataFrame,
                               atan_thr: float,
                               min_stop_pct: float,
                               max_stop_pct: float,
                               sens: float,
                               accel_cap: float) -> float:
    if len(bars) < REG_WINDOW or bars["ATR"].dropna().empty:
        return np.nan

    recent = bars["close"].iloc[-REG_WINDOW:].astype(float)
    if len(recent) < REG_WINDOW:
        return np.nan

    x = np.arange(len(recent), dtype=float)
    a, b, _c = np.polyfit(x, recent.values, 2)

    slope_norm = (2 * a * x[-1] + b) / recent.iloc[-1]
    accel = 2 * a
    atan_slope = float(np.arctan(slope_norm))

    atr = bars["ATR"].iloc[-1]
    px  = bars["close"].iloc[-1]
    atr_pct = float(atr / px) if (pd.notna(atr) and atr > 0 and px > 0) else np.nan
    if pd.isna(atr_pct):
        return np.nan

    if abs(atan_slope) > atan_thr:
        accel_factor = 1.0 + min(abs(accel), float(accel_cap))
        stop_pct = abs(slope_norm) * sens * accel_factor
        return float(np.clip(stop_pct, min_stop_pct, max_stop_pct))

    stop_pct = atr_pct * ATR_FALLBACK_MULT
    return float(np.clip(stop_pct, min_stop_pct, max_stop_pct))

def stop_hit_confirmed(sub: pd.DataFrame, is_long: bool, stop: float, confirm_bars: int) -> bool:
    k = int(max(1, confirm_bars))
    if len(sub) < k:
        return False
    closes = sub["close"].iloc[-k:].astype(float)
    return bool((closes <= stop).all()) if is_long else bool((closes >= stop).all())

def compute_initial_stop_pct(rs: pd.DataFrame,
                             entry_date: pd.Timestamp,
                             atan_thr: float,
                             max_stop: float,
                             sens: float,
                             accel_cap: float) -> float:
    cutoff = entry_date.normalize() + pd.Timedelta(hours=23, minutes=59, seconds=59)
    hist = rs[rs["_dt"] <= cutoff].copy()
    if hist.empty:
        return float(np.clip(INITIAL_MIN_STOP_PCT, INITIAL_MIN_STOP_PCT, max_stop))

    stop_pct = adaptive_stop_distance_pct(hist, atan_thr, INITIAL_MIN_STOP_PCT, max_stop, sens, accel_cap)
    if pd.notna(stop_pct):
        return float(stop_pct)

    return float(np.clip(INITIAL_MIN_STOP_PCT, INITIAL_MIN_STOP_PCT, max_stop))

def price_improvement(direction: str, new_exit: float, old_exit: float) -> float:
    d = normalize_direction(direction)
    return (new_exit - old_exit) if d == "LONG" else (old_exit - new_exit)

def simulate_trade(tr: pd.Series, rs: pd.DataFrame) -> TrailResult:
    entry_date = pd.to_datetime(tr["EntryDate"])
    exit_date  = pd.to_datetime(tr["ExitDate_Original"])
    entry_px   = float(tr["EntryPrice"])
    orig_exit_px = float(tr["ExitPrice_Original"])

    if exit_date.normalize() <= entry_date.normalize():
        return TrailResult(exit_date, orig_exit_px, "SameDayOrInvalidWindow", np.nan, np.nan)

    direction = normalize_direction(tr["Direction"])
    is_long = (direction == "LONG")

    atan_thr = float(tr["ATAN_LONG"]) if is_long else float(tr["ATAN_SHORT"])
    max_stop = float(tr["MAX_STOP_LONG"]) if is_long else float(tr["MAX_STOP_SHORT"])
    sens     = float(tr["SENS_LONG"]) if is_long else float(tr["SENS_SHORT"])
    accel_cap = float(tr["ACCEL_CAP"])

    # Initial stop at entry close (using history up to entry day)
    init_stop_pct = compute_initial_stop_pct(rs, entry_date, atan_thr, max_stop, sens, accel_cap)
    stop = entry_px * (1.0 - init_stop_pct) if is_long else entry_px * (1.0 + init_stop_pct)
    init_stop_price = float(stop)

    best_close = entry_px
    stop_source = "Initial"

    # Monitoring starts next day only
    start_dt = entry_date.normalize() + timedelta(days=1)
    end_dt   = exit_date + timedelta(days=1)

    w = rs[(rs["_dt"] >= start_dt) & (rs["_dt"] < end_dt)].copy()
    if w.empty:
        return TrailResult(exit_date, orig_exit_px, "NoRSDataInWindow", float(init_stop_pct), float(init_stop_price))

    G  = int(tr["GRACE_BARS"])
    TS = int(tr["TRAILING_START_BARS"])
    TU = int(tr["TRAILING_UPDATE_EVERY_BARS"])
    C  = int(tr["STOP_CONFIRM_BARS"])

    for i in range(len(w)):
        sub = w.iloc[:i+1]
        dt  = w.iloc[i]["_dt"]
        close = float(w.iloc[i]["close"])

        if is_long:
            best_close = max(best_close, close)
        else:
            best_close = min(best_close, close)

        # Grace period
        if i < G:
            continue

        # Stop check (close-confirm)
        if TRIGGER_ON_CLOSE:
            if stop_hit_confirmed(sub, is_long, stop, C):
                reason = "InitialSL" if stop_source == "Initial" else "TrailingSL"
                return TrailResult(dt, float(stop), reason, float(init_stop_pct), float(init_stop_price))

        # Trailing update: delayed + throttled
        if i < TS:
            continue
        every = int(max(1, TU))
        if every > 1 and (i % every) != 0:
            continue

        stop_pct = adaptive_stop_distance_pct(sub, atan_thr, MIN_STOP_PCT, max_stop, sens, accel_cap)
        if pd.isna(stop_pct):
            continue

        if is_long:
            stop = max(stop, best_close * (1.0 - stop_pct))
        else:
            stop = min(stop, best_close * (1.0 + stop_pct))

        stop_source = "Trailing"

    return TrailResult(exit_date, orig_exit_px, "OriginalExit", float(init_stop_pct), float(init_stop_price))

# ============================================================
# Summary
# ============================================================
def summarize(out: pd.DataFrame) -> pd.DataFrame:
    df = out.copy()
    df["TriggeredFlag"] = df["ExitReason"].isin(["TrailingSL", "InitialSL"])
    df["ImprovedFlag"] = df["PriceImprovement"] > 0

    def agg(g):
        n = len(g)
        trig = int(g["TriggeredFlag"].sum())
        return {
            "Trades": n,
            "SLTriggered": trig,
            "PctTriggered": trig / n if n else np.nan,
            "Triggered_PctImproved": float(g.loc[g["TriggeredFlag"], "ImprovedFlag"].mean()) if trig else np.nan,
            "AvgPctImpact_All": float(g["PctImprovement_vs_OriginalExit"].mean(skipna=True)),
            "AvgPctImpact_Triggered": float(g.loc[g["TriggeredFlag"], "PctImprovement_vs_OriginalExit"].mean(skipna=True)) if trig else np.nan,
        }

    rows = [{"Group": "ALL", **agg(df)}]
    for d, g in df.groupby(df["Direction"].astype(str).str.upper().str.strip()):
        rows.append({"Group": f"Direction={d}", **agg(g)})
    return pd.DataFrame(rows)

# ============================================================
# MAIN
# ============================================================
def main():
    # Load Step8 params (atan/max_stop)
    step8 = pd.read_csv(BEST_STEP8_PATH).iloc[0].to_dict()
    required = ["ATAN_LONG", "ATAN_SHORT", "MAX_STOP_LONG", "MAX_STOP_SHORT"]
    missing = [k for k in required if k not in step8 or pd.isna(step8.get(k))]
    if missing:
        raise ValueError(f"best_config_step8.csv missing/NaN keys: {missing}")

    # Load + build trades
    raw = read_trade_csv(TRADE_CSV_PATH)
    trades = build_trade_summary(raw)

    print("TradeIDs in year (pre-filter):", trades["TradeID"].nunique())
    print("Trades with core fields OK:", int(trades["HasAllCoreFields"].sum()))
    print("Trades with valid trailing window:", int(trades["HasValidTrailingWindow"].sum()))

    eligible = trades[(trades["HasAllCoreFields"]) & (trades["HasValidTrailingWindow"])].copy()
    if eligible.empty:
        raise RuntimeError("No eligible trades after filtering (core fields + exit>entry).")

    eligible = eligible.sort_values(["TickerNorm", "EntryDate", "TradeID"]).reset_index(drop=True)
    print("\nEligible trades to process:", len(eligible))
    print("Unique tickers in eligible:", eligible["TickerNorm"].nunique())

    # Attach params (Step8 + best combo)
    eligible["ATAN_LONG"] = float(step8["ATAN_LONG"])
    eligible["ATAN_SHORT"] = float(step8["ATAN_SHORT"])
    eligible["MAX_STOP_LONG"] = float(step8["MAX_STOP_LONG"])
    eligible["MAX_STOP_SHORT"] = float(step8["MAX_STOP_SHORT"])

    eligible["SENS_LONG"] = float(BEST["SENS_LONG"])
    eligible["SENS_SHORT"] = float(BEST["SENS_SHORT"])
    eligible["ACCEL_CAP"] = float(BEST["ACCEL_CAP"])

    eligible["GRACE_BARS"] = int(BEST["GRACE_BARS"])
    eligible["TRAILING_START_BARS"] = int(BEST["TRAILING_START_BARS"])
    eligible["TRAILING_UPDATE_EVERY_BARS"] = int(BEST["TRAILING_UPDATE_EVERY_BARS"])
    eligible["STOP_CONFIRM_BARS"] = int(BEST["STOP_CONFIRM_BARS"])

    # Preload RS15
    tickers = sorted(eligible["TickerNorm"].unique())
    print("\nPreloading RS15 data for tickers:", len(tickers))
    for i, tkr in enumerate(tickers, 1):
        prepare_rs_for_ticker(tkr)
        if i % 100 == 0:
            print(f"  Loaded {i}/{len(tickers)} tickers...")

    out_rows = []
    dbg_rows = []

    for i, tr in eligible.iterrows():
        rs = prepare_rs_for_ticker(tr["TickerNorm"])
        if rs is None or rs.empty:
            dbg_rows.append({"TradeID": tr["TradeID"], "Reason": "MissingRSData"})
            continue

        res = simulate_trade(tr, rs)

        old_exit = float(tr["ExitPrice_Original"])
        new_exit = float(res.exit_price)
        impr = price_improvement(tr["Direction"], new_exit, old_exit)

        exit_time_tr = pd.to_datetime(res.exit_time)
        exit_date_tr = exit_time_tr.normalize()
        exit_date_orig = pd.to_datetime(tr["ExitDate_Original"]).normalize()
        entry_date_norm = pd.to_datetime(tr["EntryDate"]).normalize()

        out_rows.append({
            # identifiers
            "TradeID": tr["TradeID"],
            "Currency": tr["Currency"],
            "TradeBase": tr["TradeBase"],
            "TickerNorm": tr["TickerNorm"],
            "Direction": tr["Direction"],

            # entry
            "EntryDate": tr["EntryDate"],
            "EntryDate_Norm": entry_date_norm,
            "EntryPrice": float(tr["EntryPrice"]),

            # original exit
            "ExitDate_Original": tr["ExitDate_Original"],
            "ExitDate_Original_Norm": exit_date_orig,
            "ExitPrice_Original": old_exit,

            # trailing exit
            "ExitTime_Trailing": exit_time_tr,
            "ExitDate_Trailing_Norm": exit_date_tr,
            "ExitPrice_Trailing": new_exit,
            "ExitReason": res.exit_reason,

            # comparisons
            "ExitSameDayFlag": bool(exit_date_tr == exit_date_orig),
            "ExitDateChangedFlag": bool(exit_date_tr != exit_date_orig),
            "ExitTimeChangedFlag": bool(exit_time_tr != pd.to_datetime(tr["ExitDate_Original"])),

            "HoldingDays_Original": int((exit_date_orig - entry_date_norm).days) if pd.notna(exit_date_orig) and pd.notna(entry_date_norm) else np.nan,
            "HoldingDays_Trailing": int((exit_date_tr - entry_date_norm).days) if pd.notna(exit_date_tr) and pd.notna(entry_date_norm) else np.nan,

            "PriceImprovement": impr,
            "PctImprovement_vs_OriginalExit": (impr / old_exit) if old_exit else np.nan,
            "ImprovedFlag": bool(impr > 0),

            "InitialStopPct": res.init_stop_pct,
            "InitialStopPrice": res.init_stop_price,

            # params used (for audit/repro)
            "G": int(tr["GRACE_BARS"]),
            "TS": int(tr["TRAILING_START_BARS"]),
            "TU": int(tr["TRAILING_UPDATE_EVERY_BARS"]),
            "C": int(tr["STOP_CONFIRM_BARS"]),
            "AC": float(tr["ACCEL_CAP"]),
            "SENS_LONG": float(tr["SENS_LONG"]),
            "SENS_SHORT": float(tr["SENS_SHORT"]),
        })

        dbg_rows.append({"TradeID": tr["TradeID"], "Reason": res.exit_reason})

        if (i + 1) % PRINT_EVERY == 0:
            print(f"Processed {i+1}/{len(eligible)} trades...")

    out_df = pd.DataFrame(out_rows)
    dbg_df = pd.DataFrame(dbg_rows)
    summ_df = summarize(out_df)

    reason_counts = dbg_df["Reason"].value_counts(dropna=False)
    reason_df = reason_counts.rename_axis("Reason").reset_index(name="Count")

    out_path = os.path.join(OUT_DIR, f"final_trades_{YEAR_FILTER}_COMBO_J.csv")
    summ_path = os.path.join(OUT_DIR, f"final_summary_{YEAR_FILTER}_COMBO_J.csv")
    reason_path = os.path.join(OUT_DIR, f"final_reason_counts_{YEAR_FILTER}_COMBO_J.csv")

    out_df.to_csv(out_path, index=False)
    summ_df.to_csv(summ_path, index=False)
    reason_df.to_csv(reason_path, index=False)

    print("\n✅ Saved:", out_path)
    print("✅ Saved:", summ_path)
    print("✅ Saved:", reason_path)

    print("\n=== SUMMARY ===")
    print(summ_df.to_string(index=False))

    print("\n=== EXIT REASONS ===")
    print(reason_counts.to_string())

if __name__ == "__main__":
    main()

TradeIDs in year (pre-filter): 135939
Trades with core fields OK: 135939
Trades with valid trailing window: 135785

Eligible trades to process: 135785
Unique tickers in eligible: 811

Preloading RS15 data for tickers: 811
  Loaded 100/811 tickers...
  Loaded 200/811 tickers...
  Loaded 300/811 tickers...
  Loaded 400/811 tickers...
  Loaded 500/811 tickers...
  Loaded 600/811 tickers...
  Loaded 700/811 tickers...
  Loaded 800/811 tickers...
Processed 2000/135785 trades...
Processed 4000/135785 trades...
Processed 6000/135785 trades...
Processed 8000/135785 trades...
Processed 10000/135785 trades...
Processed 12000/135785 trades...
Processed 14000/135785 trades...
Processed 16000/135785 trades...
Processed 18000/135785 trades...
Processed 20000/135785 trades...
Processed 22000/135785 trades...
Processed 24000/135785 trades...
Processed 26000/135785 trades...
Processed 28000/135785 trades...
Processed 30000/135785 trades...
Processed 32000/135785 trades...
Processed 34000/135785 trades.